# Exercicis de consolidació &mdash; Una xarxa per vigilar la xarxa

**MP 5149** &mdash; UT5 · Introducció a les xarxes neuronals supervisades &mdash; *Sessió de pràctica abans de l'examen*

*Dades: Electrical Grid Stability Simulated Data (Arzamasov, UCI, CC BY 4.0) i Fashion-MNIST
(Zalando Research, MIT). Les atribucions són a `ATRIBUCIO.md`.*

---
### Com funciona aquesta sessió

Avui no hi ha explicació: treballes tu. Aquest notebook toca tota la UT5, el NB 1.1 i el
NB 1.2, i s'assembla a l'examen. És **més llarg i una mica més difícil** que l'examen, a
propòsit: si te'n surts aquí, l'examen et semblarà assequible.

Unes quantes normes:

- Pots consultar tots els notebooks del curs i els solucionaris que tens a Classroom.
  **No pots fer servir cap eina d'intel·ligència artificial.** A l'examen tampoc no en
  tindràs, i avui és el dia de descobrir què saps fer sense ajuda.
- Compta **unes 4 hores i mitja**. Cada part porta un temps orientatiu. Si t'encalles en un
  exercici, passa al següent i torna-hi després. L'única excepció és l'exercici 4: la
  resta de la part C en depèn.
- Hi ha **bucles i funcions per escriure**, la «fontaneria» que més costa. Els primers
  porten buits `____` per omplir; els darrers els escrius sencers.
- Cada exercici té una cel·la de codi per a tu i, quan hi ha preguntes, una cel·la de
  **respostes**. Respon curt i amb números: *«l'encert puja del 81% al 94%»* val més que
  un paràgraf. Si et cal més d'una cel·la de codi, afegeix-ne.
- Els entrenaments són curts: cap cel·la no hauria de passar de 2 minuts a Colab. Les més
  llargues porten el símbol ⏱.
- La puntuació és orientativa, perquè puguis calcular com t'hauria anat si fos l'examen.

| Part | Què hi fas | Temps | Punts |
|---|---|---|---|
| A | Sense ordinador: portes lògiques, comptes i corbes de loss | 40 min | 2 |
| B | La XOR amb soroll: quantes neurones, i quanta sort | 45 min | 1,5 |
| C | La xarxa elèctrica: dades, models, neurones i el repte | 2 h 20 min | 4,5 |
| D | Imatges de roba: deu classes, pesos i errors | 50 min | 2 |

Al final hi ha cinc exercicis extra, que no puntuen, per si et sobra temps o vols practicar més a
casa.

## Preparació

La cel·la següent carrega les biblioteques i defineix cinc funcions auxiliars que ja coneixes
dels notebooks de la unitat. **No cal modificar-la**: executa-la i fes servir les funcions
quan les necessitis.

- `plot_decision_boundary(predict, X, y, grafic=None, title=None)` pinta la frontera de decisió
  d'un classificador de dues entrades, com al NB 1.1.
- `n_parameters(model)` compta els pesos i els biaixos d'un `MLPClassifier` o `MLPRegressor`
  ja entrenat, com al NB 1.2.
- `load_npz_from_url(url)` descarrega un fitxer `.npz`, com al NB 1.2.
- `show_weights(weights, grafic=None, title=None)` dibuixa 784 pesos com una imatge de 28 × 28.
- `shift_right(X, pixels)` desplaça imatges de 28 × 28 cap a la dreta.

In [ ]:
import io
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, LogisticRegression, Perceptron
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Dades de la part C (UCI, CC BY 4.0) i imatges de la part D (Fashion-MNIST, MIT)
URL_GRID = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT01-Xarxes_neuronals_supervisades/consolidacio_xarxa_electrica/xarxa_electrica.csv"
URL_FASHION = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT06-Xarxes_convolucionals/fashion_mnist/fashion_mnist.npz"

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def plot_decision_boundary(predict, X, y, grafic=None, title=None, margin=0.5, resolution=300):
    """Pinta cada regió del pla amb la classe (0 o 1) que li assigna `predict`, i els punts a sobre.

    predict : funció de predicció del model, per exemple `model.predict`.
    X, y    : mostres de dues columnes que es dibuixen damunt les regions.
    """
    grafic = grafic or plt.gca()
    X = np.asarray(X, dtype=float)
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), np.linspace(y_min, y_max, resolution))
    zz = np.asarray(predict(np.column_stack([xx.ravel(), yy.ravel()]))).reshape(xx.shape)
    grafic.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#9ecae1", "#fdae6b"], alpha=0.5)
    grafic.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=25, zorder=3)
    grafic.grid(False)
    if title:
        grafic.set_title(title)
    return grafic


def n_parameters(model):
    """Compta els pesos i els biaixos d'una xarxa de scikit-learn ja entrenada.

    Si la xarxa és dins una pipeline, passa-li l'últim pas: n_parameters(pipeline[-1]).
    """
    return sum(W.size for W in model.coefs_) + sum(b.size for b in model.intercepts_)


def load_npz_from_url(url):
    """Descarrega un fitxer .npz i l'obre amb NumPy, sense desar-lo al disc."""
    with urllib.request.urlopen(url) as response:
        return np.load(io.BytesIO(response.read()))


def show_weights(weights, grafic=None, title=None):
    """Dibuixa 784 pesos com una imatge de 28 × 28: vermell = positiu, blau = negatiu, blanc = 0."""
    grafic = grafic or plt.gca()
    image = np.asarray(weights).reshape(28, 28)
    limit = np.abs(image).max()
    grafic.imshow(image, cmap="RdBu_r", vmin=-limit, vmax=limit)
    grafic.set_xticks([])
    grafic.set_yticks([])
    grafic.grid(False)
    if title:
        grafic.set_title(title)


def shift_right(X, pixels):
    """Desplaça imatges de 784 píxels (28 × 28) `pixels` columnes cap a la dreta i omple l'esquerra de blanc."""
    images = X.reshape(-1, 28, 28)
    shifted = np.zeros_like(images)
    shifted[:, :, pixels:] = images[:, :, :28 - pixels]
    return shifted.reshape(-1, 784)

## Part A · Sense ordinador
*40 minuts · 2 punts*

En aquesta part no cal executar res. Agafa paper i llapis, i respon a les cel·les de respostes.

### Exercici 1 · La XNOR *(1 punt)*

La porta **XNOR** és la contrària de la XOR: val 1 quan les dues entrades són **iguals**.

| x1 | x2 | XNOR |
|---|---|---|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Per construir-la pots fer servir aquestes quatre portes, i cadascuna la fa una sola neurona. L'AND
és a la secció 3.1 del NB 1.1, i l'OR i el NAND, a la secció 5. El NOR és nou: és el contrari de
l'OR.

| x1 | x2 | AND | OR | NAND | NOR |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 1 | 1 |
| 0 | 1 | 0 | 1 | 1 | 0 |
| 1 | 0 | 0 | 1 | 1 | 0 |
| 1 | 1 | 1 | 1 | 0 | 0 |

- **AND**: són 1 totes dues?
- **OR**: n'hi ha almenys una que sigui 1?
- **NAND** (no AND): que **no** siguin 1 totes dues.
- **NOR** (no OR): que **cap** no sigui 1, és a dir, que totes dues siguin 0.

Recorda com decideix un perceptró (NB 1.1, secció 2.1): calcula $z = w_1 x_1 + w_2 x_2 + b$ i,
amb la funció step, respon 1 si $z \geq 0$ i 0 si no.

1. Dibuixa els quatre punts en un paper, amb un color per a cada classe. Un perceptró tot sol
   pot aprendre la XNOR? Per què?
2. Construeix una xarxa amb **dues neurones ocultes i una de sortida**, on cada neurona fa una
   de les portes AND, OR, NAND o NOR (pots repetir porta). Digues quina porta fa cada neurona i
   completa la taula de la cel·la de respostes. Compara-la amb la xarxa de la XOR de la
   secció 5 del NB 1.1: què canvia?
3. Tria uns pesos $w_1$, $w_2$ i un biaix $b$ perquè una neurona faci la porta **NOR** (val 1
   només quan les dues entrades són 0). Comprova-ho amb una taula amb la $z$ de cada fila.

**Respostes**

1.
2. $h_1$ = ______, $h_2$ = ______, sortida = ______

   | x1 | x2 | h1 | h2 | sortida | XNOR |
   |---|---|---|---|---|---|
   | 0 | 0 |  |  |  | 1 |
   | 0 | 1 |  |  |  | 0 |
   | 1 | 0 |  |  |  | 0 |
   | 1 | 1 |  |  |  | 1 |

   Què canvia respecte a la XOR:
3. $w_1$ = ____, $w_2$ = ____, $b$ = ____

   | x1 | x2 | z | sortida |
   |---|---|---|---|
   | 0 | 0 |  |  |
   | 0 | 1 |  |  |
   | 1 | 0 |  |  |
   | 1 | 1 |  |  |

### Exercici 2 · Comptar i llegir *(1 punt)*

1. **Paràmetres.** Una xarxa té 12 entrades, una capa oculta de 16 neurones i 1 neurona de
   sortida (12-16-1). Quants pesos i quants biaixos té? Quants paràmetres en total? Fes el
   mateix per a una xarxa 784-128-64-10. *Ho comprovaràs amb codi a l'exercici 6.*
2. **Passes per epoch.** Entrenes amb 6.000 files. Quantes passes, és a dir, actualitzacions
   dels pesos, fa una epoch amb `batch_size=200`, el valor per defecte? I amb 64? I amb 6.000?
   Quantes passes fa en total un entrenament de 50 epochs amb batches de 200?
3. **Softmax i loss.** Una xarxa que reconeix peces de roba dona aquestes probabilitats per a
   tres imatges (són inventades):

   | Imatge | Classe real | Probabilitat que dona a la classe real | Classe que prediu (probabilitat) |
   |---|---|---|---|
   | A | sabatilla | 0,92 | sabatilla (0,92) |
   | B | camisa | 0,30 | samarreta (0,55) |
   | C | jersei | 0,02 | abric (0,95) |

   Quines prediccions són correctes? Ordena les tres imatges de menys loss a més loss.
   Per què la loss castiga tant la C?
4. **Corbes de loss.** Les quatre corbes de la figura són **la mateixa xarxa amb les mateixes
   dades**; només canvia el learning rate. Relaciona cada corba amb un dels quatre casos de
   la vall de la secció 7 del NB 1.1 (massa petit, adequat, gran, massa gran) i explica què hi
   veus que t'ho fa dir.

![Quatre corbes de loss amb learning rates diferents](https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT01-Xarxes_neuronals_supervisades/consolidacio_xarxa_electrica/corbes_learning_rate.png)

**Respostes**

1. 12-16-1:
   784-128-64-10:
2.
3.
4. A: · B: · C: · D:

## Part B · La XOR amb soroll
*45 minuts · 1,5 punts*

### Exercici 3 · Quantes neurones, i quanta sort *(1,5 punts)*

La XOR del NB 1.1 tenia quatre punts. Aquí en farem una versió amb 600: punts repartits per tot
el quadrat de −1 a 1, de classe 1 quan $x_1$ i $x_2$ tenen **signe diferent** i de classe 0
quan el tenen igual. És a dir, quadrants de colors alterns. Una mica de soroll fica alguns punts
al quadrant del costat. La cel·la ja està feta: executa-la.

Les dues columnes ja són centrades al voltant del zero i tenen la mateixa escala, com les
llunes del NB 1.1, així que aquí no cal escalar.

In [ ]:
rng = np.random.default_rng(42)
X_xor = rng.uniform(-1, 1, size=(600, 2))
y_xor = ((X_xor[:, 0] > 0) != (X_xor[:, 1] > 0)).astype(int)   # 1 si x1 i x2 tenen signe diferent
X_xor = X_xor + rng.normal(0, 0.08, size=X_xor.shape)          # soroll: alguns punts canvien de quadrant

X_train, X_test, y_train, y_test = train_test_split(
    X_xor, y_xor, test_size=0.25, random_state=42, stratify=y_xor)

plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", edgecolor="k", s=20)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title(f"XOR amb soroll: {len(X_train)} d'entrenament, {len(X_test)} de test")
plt.show()

1. Entrena un `Perceptron(random_state=42)` i un
   `MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", learning_rate_init=0.01, max_iter=2000, random_state=42)`.
   Calcula l'encert de test de cadascun i dibuixa les dues fronteres l'una al costat de l'altra
   amb `plot_decision_boundary` (un `plt.subplots(1, 2)`, com al NB 1.1).
2. A la secció 9 del NB 1.1 vas veure que la mateixa xarxa pot sortir bé o encallar-se segons
   la llavor. Ara ho comptarem. Completa els buits `____` del bucle: per a cada mida de la capa
   oculta, entrena 20 xarxes amb les llavors de 0 a 19 i compta quantes arriben a un **90%
   d'encert a test**.

   > ⏱ Són 100 entrenaments: uns 15 segons en un ordinador i menys d'un minut a Colab.
3. Respon:
   - Quin encert fa el perceptró? Per què?
   - Amb 1 i 2 neurones ocultes, cap llavor no arriba al 90%. A la XOR de quatre punts del
     NB 1.1, però, dues neurones n'hi havia prou. Dibuixa la frontera d'una xarxa de 2
     neurones (amb qualsevol llavor) i explica què li falta.
   - Amb 3 i 4 neurones, unes llavors hi arriben i d'altres no. Per què?
   - Quina conclusió en treus sobre l'amplada de la capa oculta?

In [ ]:
# El teu codi aquí

El bucle de l'apartat 2. Omple els `____` i executa'l:

In [ ]:
sizes = [1, 2, 3, 4, 8]
rows = []
for n_hidden in ____:
    n_ok = 0                                    # quantes llavors arriben al 90% amb aquesta mida
    for seed in range(20):
        model = MLPClassifier(hidden_layer_sizes=(____,), activation="tanh", learning_rate_init=0.01,
                              max_iter=2000, random_state=____)
        model.fit(X_train, y_train)
        if model.score(X_test, y_test) >= 0.9:
            n_ok = ____
    rows.append({"neurones ocultes": n_hidden, "llavors que arriben al 90% (de 20)": ____})

pd.DataFrame(rows)

In [ ]:
# El teu codi aquí

**Respostes**

- El perceptró:
- 1 i 2 neurones:
- 3 i 4 neurones:
- Conclusió:

## Part C · La xarxa elèctrica
*2 hores i 20 minuts · 4,5 punts*

### El cas

Una xarxa elèctrica ha d'equilibrar a cada moment el que es produeix amb el que es consumeix.
Una idea per fer-ho sense una central que ho controli tot és que **el preu de l'electricitat
canviï segons com va la xarxa**: quan sobra electricitat, baixa el preu i els consumidors en
gasten més; quan en falta, puja i en gasten menys. Se'n diu control descentralitzat
(*Decentral Smart Grid Control*).

El risc és que, si els participants reaccionen **massa tard** o **massa fort**, la xarxa comenci
a oscil·lar, com un gronxador que algú empeny a destemps, i **es desestabilitzi**.

Un investigador del Karlsruhe Institute of Technology, Vadim Arzamasov, va simular 10.000
configuracions d'una xarxa petita amb forma d'estrella: un **productor** al centre (node 1) i
**tres consumidors** (nodes 2, 3 i 4). Per a cada configuració, la simulació diu si la xarxa és
estable. El problema és que la simulació costa de fer, i l'operador voldria un model que ho
digués a l'instant a partir dels paràmetres de cada participant. Aquest model serà una xarxa
neuronal: **una xarxa per vigilar la xarxa**.

Les dades són **simulades** amb un model físic, no mesurades en una xarxa real.

### Les columnes

| Columna | Què és |
|---|---|
| `tau1` … `tau4` | **temps de reacció** de cada participant, en segons (de 0,5 a 10): quant tarda a ajustar la producció o el consum quan canvia el preu. L'1 és el productor |
| `p1` … `p4` | **potència**: positiva per al productor, que la injecta, i negativa per als consumidors, que la gasten |
| `g1` … `g4` | **elasticitat al preu** (de 0,05 a 1): com de fort reacciona cada participant quan canvia el preu |
| `stab` | resultat numèric de la simulació: com més positiu, més inestable |
| `stabf` | resultat de la simulació en text: `stable` o `unstable` |

Els noms venen de les lletres gregues de l'article original: `tau` és τ, la lletra que es fa
servir en física per als temps de reacció, i `g` és γ (gamma). `p` és la potència.

> ⚠️ **En aquest fitxer hi ha una fuga d'informació.** Una de les columnes no es pot fer servir
> per predir, encara que el model hi surti guanyant. Per trobar-la, fes-te aquesta pregunta per
> a cada columna: **si ja tinguéssim aquesta dada, encara ens caldria fer la simulació?**

La cel·la següent carrega el fitxer. Ja està feta: només l'has d'executar.

In [ ]:
df = pd.read_csv(URL_GRID)
df.head()

### Exercici 4 · Primer cop d'ull i partició *(1 punt)*

1. Quantes configuracions i quantes columnes té el fitxer? Quina és la variable objectiu? És
   un problema de regressió o de classificació? Quin percentatge de configuracions són estables?
2. Quina columna és la fuga? Per què no es pot fer servir?
3. Calcula `p1 + p2 + p3 + p4` per a totes les files i mira'n el valor més gran en valor
   absolut. Què vol dir, pensant en una xarxa elèctrica? La columna `p1` aporta alguna
   informació que no tinguin les altres tres?
4. Prepara les dades. Totes les cel·les de la part C depenen d'aquest apartat, i els noms han
   de ser exactament aquests:
   - Crea una columna **`stable`** que valgui 1 si `stabf` és `"stable"` i 0 si no.
   - Parteix `df` amb `train_test_split`, igual que al NB 2.2 del 5134: primer aparta un 20%
     per al **test**, i després parteix la resta en **entrenament** (75%) i **validació** (25%).
     Fes servir `random_state=42` i **estratifica** totes dues particions per la columna
     `stable` (`stratify=...`, com al NB 1.1). Els tres DataFrames s'han de dir **`train`**,
     **`valid`** i **`test`**. El test queda segellat fins a l'exercici 7.
   - Crea una llista **`FEATURES`** amb els noms de les **12 columnes d'entrada**: les de `tau`,
     les de `p` i les de `g`.
   - Entrena un `DummyClassifier()` amb `train` i calcula'n l'encert de validació.

> **Comprovació.** Has d'obtenir 6.000 configuracions a `train`, 2.000 a `valid` i 2.000 a
> `test`, `len(FEATURES)` ha de ser 12 i el `DummyClassifier` ha d'encertar un **0,638** a
> validació.
>
> ⚠️ **Aquest exercici condiciona la resta de la part C.** No continuïs fins que la comprovació
> et quadri.

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí: la columna stable, la partició, FEATURES i el Dummy

**Respostes**

1.
2.
3.

### Exercici 5 · Una neurona contra una xarxa *(1 punt)*

A partir d'ara compararàs molts models. Per no repetir les mateixes línies cada vegada, completa
la funció **`try_model`**: entrena un model amb unes columnes de `train` i en torna una fila de
la taula. Omple els buits `____` i comprova-la amb la cel·la de sota, que ha de donar un 0,638 a
entrenament i a validació.

A tots els models que no siguin el Dummy, escala les entrades amb una **pipeline**, com a la
secció 3.2 del NB 1.1: `make_pipeline(StandardScaler(), model)`. Després fes una taula amb:

1. `Perceptron(random_state=42)`;
2. `LogisticRegression()`, la **regressió logística**, que ja vas veure als NB 1.1 i 1.2: una sola
   neurona amb la sigmoide a la sortida (la treballareu a fons a la UT4 del 5134);
3. `MLPClassifier(hidden_layer_sizes=(32,), activation="identity", max_iter=500, random_state=42)`,
   una xarxa de 32 neurones **sense funció d'activació**;
4. el mateix MLP amb l'activació per defecte, la ReLU (treu el paràmetre `activation`).

Pots fer la taula com al NB 1.1: una llista de files i `pd.DataFrame(rows).set_index("model")`.

Respon:

1. Quants punts d'encert guanya cada model respecte al Dummy?
2. El perceptró i la regressió logística són una sola neurona. Què et diuen els seus encerts
   sobre aquest problema?
3. L'MLP de 32 neurones sense activació encerta pràcticament igual que la regressió logística.
   Per què? (NB 1.1, secció 10.3.)
4. Dins `try_model`, la pipeline fa `fit` amb `train`. Què aprèn el `StandardScaler` en aquest
   `fit`, i amb quines dades? Què passaria si escaléssim `df` sencer **abans** de partir-lo?
5. Afegeix a la taula un `MLPClassifier(hidden_layer_sizes=(16,), max_iter=500, random_state=42)`
   escalat que faci servir `FEATURES + ["stab"]`. Quin encert fa? Per què no el podem fer servir?

In [ ]:
def try_model(name, model, columns):
    """Entrena `model` amb les `columns` de train i el mesura a entrenament i a validació.

    Retorna un diccionari amb el nom, l'encert d'entrenament, l'encert de validació i els
    segons que ha tardat a entrenar. Exemple: try_model("Dummy", DummyClassifier(), FEATURES)
    """
    start = time.perf_counter()
    model.fit(train[____], train["stable"])
    seconds = time.perf_counter() - ____
    return {"model": name,
            "encert entrenament": model.score(____, ____),
            "encert validació": model.score(valid[columns], ____),
            "temps (s)": seconds}

In [ ]:
# Comprovació: ha de sortir un 0,638 a entrenament i a validació
try_model("Dummy", DummyClassifier(), FEATURES)

In [ ]:
# El teu codi aquí

**Respostes**

1.
2.
3.
4.
5.

### Exercici 6 · Quantes neurones? *(1 punt)*

Ara escriu el bucle sencer, sense buits.

1. Per a cada mida de la llista de sota, entrena
   `make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=size, max_iter=500, random_state=42))`
   amb `FEATURES` i guarda la fila de `try_model`. Afegeix a cada fila una columna
   `"paràmetres"` amb `n_parameters(model[-1])`: `model[-1]` és l'últim pas de la pipeline, la
   xarxa. Comprova que la xarxa de mida `(16,)` té els paràmetres que vas calcular a
   l'exercici 2.

   ```python
   sizes = [(1,), (2,), (4,), (8,), (16,), (32,), (64,), (128,), (128, 128)]
   ```

   > ⏱ Uns 25 segons en un ordinador i poc més d'un minut a Colab.
2. Dibuixa en un mateix gràfic l'encert d'entrenament i el de validació contra el nombre de
   paràmetres: dues línies amb `plt.plot`, una `label` per a cadascuna i `plt.legend()`. Posa
   l'eix x en escala logarítmica amb `plt.xscale("log")`, perquè els paràmetres van de pocs a
   milers.
3. Amb **poques dades**: entrena les mides `(4,)`, `(16,)` i `(128, 128)`, ara amb
   `max_iter=2000`, només amb les 300 primeres files de l'entrenament (`train.head(300)`), i
   mesura-les a entrenament (amb aquestes 300 files) i a validació. Aquí no pots fer servir
   `try_model`, perquè entrena amb `train` sencer: escriu el `fit` i els `score` a mà.

Respon:

1. Quina mida triaries, i per què? Fixa't també en els paràmetres.
2. De 4 a 8 neurones l'encert de validació **baixa**. Com pot ser, si 8 neurones poden fer tot
   el que fan 4? (Pensa en l'exercici 3.)
3. Hi ha sobreajust amb les 6.000 files? I amb les 300? Justifica-ho amb números.

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

**Respostes**

1.
2.
3.

### Exercici 7 · El repte i el test *(1,5 punts)*

L'operador de la xarxa vol un model que encerti tant com sigui possible. Com més amunt deixis
l'encert de **validació**, millor:

| Nivell | Encert de validació | Punts |
|---|---|---|
| 🥉 Bronze | **93%** o més | 0,25 |
| 🥈 Plata | **95,5%** o més | 0,5 |
| 🥇 Or | **96,8%** o més | 0,75 |

Les regles del joc:

- El model ha de ser un **`MLPClassifier` dins una pipeline amb `StandardScaler`**, amb
  `random_state=42`. Pots canviar tota la resta: neurones, capes, activació, `max_iter`,
  `alpha` (NB 1.1, exercici 7)... **i les columnes**.
- Entrenes només amb `train` i compares amb `valid`. El test no es toca, i `stab` no hi pot ser.

Fes una taula amb **tots els intents** (com a mínim tres) i marca quin és el teu model final.

Abans de començar, fes aquest experiment, que et donarà pistes: entrena
`MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)`, escalat, amb cinc
grups de columnes: només les `tau`, només les `p`, només les `g`, les `tau` i les `g` juntes, i
totes (`FEATURES`). Un diccionari `{nom del grup: llista de columnes}` i un bucle et van bé.

Respon:

1. Quin nivell has aconseguit, i amb quin model?
2. Què diu l'experiment dels grups de columnes? Com pot ser que **treure** columnes faci que el
   model encerti **més**?
3. **Obre el test**, una sola vegada, amb el teu model final tal com està: quin encert fa?
   S'assembla al de validació? Calcula també el del `DummyClassifier` sobre el test, i escriu en
   dues o tres frases el missatge que enviaries a l'operador de la xarxa.

<details>
<summary><b>Pistes</b> (obre-les només si les necessites)</summary>

- La plata surt de l'exercici 6.
- Per a l'or, mira bé l'experiment dels grups de columnes: no totes les columnes ajuden.

</details>

In [ ]:
# El teu codi aquí: l'experiment dels grups de columnes

In [ ]:
# El teu codi aquí: els intents del repte

In [ ]:
# El teu codi aquí: el test, una sola vegada

**Respostes**

1.
2.
3.

## Part D · Imatges de roba
*50 minuts · 2 punts*

**Fashion-MNIST** són 70.000 fotografies de peces de roba de la botiga en línia Zalando, en el
mateix format que MNIST: 28 × 28 píxels en escala de grisos i 10 classes. Es va publicar el 2017
perquè MNIST ja era massa fàcil. Ja el coneixes de l'exercici 8 del NB 1.2.

Per anar de pressa, farem servir **20.000 imatges d'entrenament** i les **10.000 últimes** del
conjunt d'entrenament com a validació. El test no el farem servir. Els píxels es divideixen per
255, com al NB 1.2.

> ⏱ La descàrrega són uns 30 MB: uns segons a Colab.

In [ ]:
fashion = load_npz_from_url(URL_FASHION)
class_names = fashion["class_names"]
X_fashion = fashion["X_train"].astype(np.float32) / 255.0
y_fashion = fashion["y_train"]

Xf_train, yf_train = X_fashion[:20_000], y_fashion[:20_000]
Xf_val, yf_val = X_fashion[50_000:], y_fashion[50_000:]
print("Entrenament:", Xf_train.shape, "  Validació:", Xf_val.shape)
print("Classes:", list(class_names))

fig, grafics = plt.subplots(2, 10, figsize=(14, 3.4))
for grafic, image, label in zip(grafics.ravel(), Xf_train, yf_train):
    grafic.imshow(image.reshape(28, 28), cmap="gray_r")
    grafic.set_title(class_names[label], fontsize=9)
    grafic.axis("off")
plt.tight_layout()
plt.show()

### Exercici 8 · Deu classes *(2 punts)*

1. **La taula.** Escriu una funció semblant a `try_model` que treballi amb `Xf_train`, `yf_train`,
   `Xf_val` i `yf_val`, aquesta vegada sense cap buit, i fes una taula amb:
   - `DummyClassifier()`;
   - `LogisticRegression(C=0.01, max_iter=300)`, la mateixa del NB 1.2, secció 6.2;
   - `RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)`;
   - `MLPClassifier(hidden_layer_sizes=(128,), max_iter=15, random_state=42)`.

   Aquí no cal la pipeline amb `StandardScaler`: els píxels ja van de 0 a 1. Guarda la xarxa
   en una variable, perquè la faràs servir a la resta de l'exercici.

   > ⏱ Uns 15 segons en un ordinador i menys d'un minut a Colab.
2. **La xarxa per dins.** Quants paràmetres té la xarxa? Fes el càlcul a mà, com a l'exercici 2,
   i comprova'l amb `n_parameters`. Quina funció d'activació té la capa de sortida
   (`out_activation_`), i per què?
3. **Els pesos.** Dibuixa els pesos de les 16 primeres neurones ocultes amb `show_weights`, en
   una graella de 2 × 8 (la columna `i` de `coefs_[0]` són els 784 pesos de la neurona `i`).
   S'assemblen a peces de roba senceres?
4. **Els errors.** Dibuixa la matriu de confusió de la xarxa a validació amb
   `ConfusionMatrixDisplay.from_predictions(..., display_labels=class_names, xticks_rotation="vertical", ax=grafic, colorbar=False)`,
   en un gràfic de 8 × 7. Quina és la confusió més freqüent? Té sentit, mirant les fotos?
5. Tria una imatge de validació mal classificada i mira'n les deu probabilitats amb
   `predict_proba`. La xarxa estava segura o dubtava?

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

**Respostes**

1.
2.
3.
4.
5.

## Exercicis extra
*Si t'ha sobrat temps, o per practicar més a casa. No puntuen.*

Els tres primers continuen la part C, amb la xarxa elèctrica, i els dos últims, la part D, amb la
roba.

### Exercici 9 · El learning rate

Entrena aquesta xarxa amb el learning rate de la llista, i dibuixa **les sis corbes de loss al
mateix gràfic**, amb una `label` que digui el learning rate i l'encert de validació:

```python
make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), solver="sgd", momentum=0,
                                              learning_rate_init=lr, max_iter=200, n_iter_no_change=200,
                                              random_state=42))
learning_rates = [0.0001, 0.001, 0.01, 0.1, 1, 10]
```

`solver="sgd"` amb `momentum=0` és el gradient descent pur, com a la secció 10.2 del NB 1.1. La
corba de loss és a l'atribut `loss_curve_` de la xarxa, que dins la pipeline és `model[-1]`.
Posa `plt.ylim(0, 1)` perquè es vegin bé totes.

Respon: a quin dels quatre casos de la vall correspon cada learning rate? Quin triaries?

In [ ]:
# El teu codi aquí

**Resposta:**

### Exercici 10 · Qui fabrica les característiques?

Al repte (exercici 7), la xarxa arribava a l'or amb les `tau` i les `g`. Ara prova un model que
**no és una xarxa**: una regressió logística que rep columnes noves fabricades abans per una
`PolynomialFeatures`.

La `PolynomialFeatures` de grau 3 multiplica les columnes que té, de totes les maneres possibles
fins a tres factors: els quadrats i els cubs de cada columna (`tau1²`, `tau1³`) i els productes
entre columnes (`tau1 × g1`, `tau1 × g1 × g2`...). Amb 2 columnes, `a` i `b`, en fabricaria 10:
`1`, `a`, `b`, `a²`, `ab`, `b²`, `a³`, `a²b`, `ab²` i `b³`.

```python
make_pipeline(StandardScaler(), PolynomialFeatures(degree=3), LogisticRegression(max_iter=3000))
```

Entrena-la amb les `tau` i les `g`. Quantes columnes fabrica la `PolynomialFeatures` (són a
`model[1].n_output_features_`)? Quin encert fa a validació? Compara-ho amb la teva xarxa del
repte: qui ha preparat les característiques en cada cas?

In [ ]:
# El teu codi aquí

**Resposta:**

### Exercici 11 · Predir un número

`stab` no es pot fer servir per predir `stabf`, però sí que es pot **predir**: és un número, i
per tant és una regressió. Entrena una `LinearRegression` i una
`make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=1000, random_state=42))`
que prediguin `stab` a partir de `FEATURES`, i compara'n el R2 de validació (el `score` d'un
model de regressió és el R2).

*Pista:* els valors de `stab` són molt petits (entre −0,08 i 0,11). Amb números tan petits, la
loss de la xarxa també ho és, i l'entrenament s'atura de seguida perquè «ja no millora». Prova a
entrenar la xarxa amb `stab` multiplicada per 100, i mesura-la també amb `stab` per 100.

In [ ]:
# El teu codi aquí

**Resposta:**

### Exercici 12 · Batches i límits

1. **Mini-batches.** Per a cada `batch_size` de la llista, crea
   `MLPClassifier(hidden_layer_sizes=(128,), batch_size=batch_size, random_state=42)` i entrena'l
   **5 epochs** amb `partial_fit`: cada crida fa una epoch, i la primera vegada cal passar-li
   `classes=np.arange(10)`. Després de cada epoch, guarda l'encert de validació en una llista.
   Dibuixa les tres llistes en un mateix gràfic, amb l'epoch a l'eix x.

   ```python
   batch_sizes = [20_000, 1024, 64]
   ```

   Quantes passes fa cada xarxa en una epoch? I en total? Quina aprèn més en les mateixes 5
   epochs, i per què, si totes han vist exactament les mateixes imatges?
2. **Desplaçar les imatges.** Fes una taula amb l'encert de validació de la regressió logística,
   el random forest i la xarxa de l'exercici 8 quan les imatges de validació es desplacen 0, 1,
   2, 3 i 4 píxels cap a la dreta amb `shift_right`, com a la secció 10 del NB 1.2. Dibuixa-ho:
   una línia per model. Quant perd cada model amb 3 píxels? Per què? Quin tipus de xarxa ho
   resoldria?

   > ⏱ Tot l'exercici, uns 15 segons en un ordinador i menys d'un minut a Colab.

In [ ]:
# El teu codi aquí

In [ ]:
# El teu codi aquí

**Respostes**

1.
2.

### Exercici 13 · Aturar a temps

Entrena la xarxa de Fashion-MNIST de l'exercici 8 amb `max_iter=60`, `early_stopping=True`,
`validation_fraction=0.1` i `n_iter_no_change=3` (NB 1.2, exercici 5). En quina epoch s'atura
(`n_iter_`)? Quin encert fa a validació, comparat amb el de les 15 epochs de l'exercici 8?
Dibuixa en dos gràfics, l'un al costat de l'altre, la corba de loss (`loss_curve_`) i l'encert de
la validació interna (`validation_scores_`).

In [ ]:
# El teu codi aquí

**Resposta:**

---
### Quan acabis

Desa el notebook al teu Drive. Quan es pengi el solucionari a Classroom, compara-hi les teves
respostes, sobretot les dels exercicis on has dubtat: és allà on hi ha la feina de cara a
l'examen.